In [1]:
import os

# Sesuaikan nama file CSV kamu
file_path = '../data/raw/2019-oct.csv'

file_size = os.path.getsize(file_path) / (1024**3)
print(f"Ukuran file: {file_size:.2f} GB")

Ukuran file: 5.28 GB


In [2]:
import pandas as pd

df_peek = pd.read_csv(file_path, nrows=5)
print("Kolom:", df_peek.columns.tolist())
print("\nSample data:")
display(df_peek)

Kolom: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Sample data:


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [3]:
# Proses ini ~3-5 menit, biarkan jalan sampai selesai
total_rows = 0
chunk_size = 500_000

for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    total_rows += len(chunk)
    print(f"Processed: {total_rows:,} rows...", end='\r')

print(f"\n Total rows: {total_rows:,}")

Processed: 42,448,764 rows...
 Total rows: 42,448,764


In [4]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import time

file_path = '../data/raw/2019-oct.csv'
output_path = '../data/processed/ecommerce.parquet'

dtypes = {
    'event_type'    : 'category',
    'product_id'    : 'int32',
    'category_id'   : 'int64',
    'category_code' : 'category',
    'brand'         : 'category',
    'price'         : 'float32',
    'user_id'       : 'int32',
    'user_session'  : 'str'
}

chunk_size    = 500_000
parquet_writer = None
start_time    = time.time()
total_rows    = 0

print("Mulai konversi CSV → Parquet...\n")

for i, chunk in enumerate(pd.read_csv(
    file_path,
    chunksize=chunk_size,
    dtype=dtypes,
    parse_dates=['event_time']
)):
    table = pa.Table.from_pandas(chunk)

    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression='snappy'
        )

    parquet_writer.write_table(table)
    total_rows += len(chunk)

    elapsed = time.time() - start_time
    pct     = (total_rows / 42_448_764) * 100
    print(f"Chunk {i+1:>2} | {total_rows:>12,} rows | {pct:>5.1f}% | {elapsed:>5.0f}s", end='\r')

if parquet_writer:
    parquet_writer.close()

# Laporan hasil
original_size   = os.path.getsize(file_path) / (1024**3)
compressed_size = os.path.getsize(output_path) / (1024**3)
ratio           = (1 - compressed_size / original_size) * 100

print(f"\n\n Konversi selesai!")
print(f"   CSV asli     : {original_size:.2f} GB")
print(f"   Parquet baru : {compressed_size:.2f} GB")
print(f"   Kompresi     : {ratio:.0f}% lebih kecil")
print(f"   Total rows   : {total_rows:,}")

Mulai konversi CSV → Parquet...

Chunk 85 |   42,448,764 rows | 100.0% |   854s

 Konversi selesai!
   CSV asli     : 5.28 GB
   Parquet baru : 1.25 GB
   Kompresi     : 76% lebih kecil
   Total rows   : 42,448,764


In [5]:
import duckdb

con = duckdb.connect('../data/processed/ecommerce.duckdb')

con.execute("""
    CREATE VIEW IF NOT EXISTS ecommerce AS
    SELECT * FROM read_parquet('../data/processed/ecommerce.parquet')
""")

result = con.execute("""
    SELECT
        COUNT(*)                    AS total_transactions,
        COUNT(DISTINCT user_id)     AS unique_users,
        COUNT(DISTINCT product_id)  AS unique_products,
        COUNT(DISTINCT brand)       AS unique_brands,
        ROUND(AVG(price), 2)        AS avg_price,
        MIN(event_time)             AS earliest_date,
        MAX(event_time)             AS latest_date
    FROM ecommerce
""").df()

print("Dataset Overview:")
display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset Overview:


,total_transactions,unique_users,unique_products,unique_brands,avg_price,earliest_date,latest_date
0,42448764,3022290,166794,3444,290.32,2019-10-01 07:00:00+07:00,2019-11-01 06:59:59+07:00


In [6]:
quality = con.execute("""
    SELECT
        COUNT(*)                            AS total_rows,

        -- Null counts
        COUNT(*) - COUNT(event_time)        AS null_event_time,
        COUNT(*) - COUNT(product_id)        AS null_product_id,
        COUNT(*) - COUNT(category_id)       AS null_category_id,
        COUNT(*) - COUNT(category_code)     AS null_category_code,
        COUNT(*) - COUNT(brand)             AS null_brand,
        COUNT(*) - COUNT(price)             AS null_price,
        COUNT(*) - COUNT(user_id)           AS null_user_id,
        COUNT(*) - COUNT(user_session)      AS null_user_session,

        -- Anomali harga
        COUNT(*) FILTER (WHERE price <= 0)  AS zero_or_neg_price,
        COUNT(*) FILTER (WHERE price > 5000) AS extreme_price

    FROM ecommerce
""").df()

print("🔍 Data Quality Report:")
display(quality.T.rename(columns={0: 'count'}))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🔍 Data Quality Report:


,count
total_rows,42448764
null_event_time,0
null_product_id,0
null_category_id,0
null_category_code,13515609
null_brand,6117080
null_price,0
null_user_id,0
null_user_session,2
zero_or_neg_price,68673


In [7]:
sample = con.execute("""
    SELECT * FROM ecommerce
    USING SAMPLE 1 PERCENT (bernoulli)
""").df()

sample.to_csv('../data/sample/ecommerce_sample_1pct.csv', index=False)

sample_size = os.path.getsize('../data/sample/ecommerce_sample_1pct.csv') / (1024**2)
print(f"Sample saved!")
print(f"   Rows : {len(sample):,}")
print(f"   Size : {sample_size:.1f} MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Sample saved!
   Rows : 424,115
   Size : 55.2 MB


In [8]:
con.close()
print("Connection closed")

Connection closed
